In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb==0.17.5", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb==0.17.5", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


In [2]:
# ===== NEW CELL — run first, immediately after the restart from cell 0 =====
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"])

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
%%writefile model_7e.py
##################################################
# Defines HELM Phase 13A: Elastic Threshold Router
# Has a total of 32 heads, d_head = 64; only 16 will be used at a time
# True Decoupling of d_model = d_head * n_head
# Acheived via expansion layer
# Target 8 16 32
# 4 Perm heads
# No Dead Head Penalty
# Still Use Clamp
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm_7e_direct_rawrank"

    def __init__(
        self,
        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 32,
        d_head = 64,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # HELM_7e Router
        num_router_latents = 4,
        num_permanent_heads = 8,
        head_target_min = 8,
        head_target_center = 16,
        head_target_max = 32,
        easiness_cdf_breakpoints = None,
        count_loss_lambda = 0.5,
        router_grad_clip = 0.05,

        # Raw-rank regularization (Direct executed-context spectral participation-rank regularization)
        rank_loss_lambda = 0.25,
        rank_floor_scale = 0.80,
        rank_sample_tokens = 64,
        rank_warmup_start_frac = 0.02,
        rank_warmup_steps_frac = 0.08,
        rank_reference_ratios = None,

        # Permanent-head training noise
        jitter_noise = 0.01,

        # nGPT self attention and FFN hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        dataset_total_steps = 65000,
        **kwargs
    ):
        # General model
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization / MLM
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # HELM_7e Router
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.count_loss_lambda = count_loss_lambda
        self.router_grad_clip = router_grad_clip

        # Rank regularizer. The reference curve comes from the user's Dense-32
        # checkpoint-006500 audit. We use only 80% of that healthy dense profile
        # as a FLOOR, not as an equality target.
        self.rank_loss_lambda = float(rank_loss_lambda)
        self.rank_floor_scale = float(rank_floor_scale)
        self.rank_sample_tokens = int(rank_sample_tokens)
        self.rank_warmup_start_frac = float(rank_warmup_start_frac)
        self.rank_warmup_steps_frac = float(rank_warmup_steps_frac)
        if rank_reference_ratios is None:
            rank_reference_ratios = [0.6620212938, 0.6022618308, 0.4980440646, 0.6409693696, 0.6126576987, 0.576476318, 0.6936327685, 0.6572240894, 0.6555443588, 0.685528013, 0.6417101392, 0.5373020249]
        self.rank_reference_ratios = [float(x) for x in rank_reference_ratios]

        self.jitter_noise = jitter_noise

        elastic = num_attention_heads - num_permanent_heads
        if num_permanent_heads != head_target_min:
            raise ValueError(
                "HELM_7e uses permanent heads as the structural minimum; "
                "num_permanent_heads must equal head_target_min."
            )
        if elastic <= 0:
            raise ValueError("HELM_7e requires at least one elastic head")
        if not (head_target_min <= head_target_center <= head_target_max <= num_attention_heads):
            raise ValueError("Invalid HELM head targets")
        if len(self.rank_reference_ratios) != num_hidden_layers:
            raise ValueError(
                f"rank_reference_ratios must have one entry per layer "
                f"({num_hidden_layers}), got {len(self.rank_reference_ratios)}"
            )

        # nGPT
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.dataset_total_steps = dataset_total_steps

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# HELM_7e multi-latent router
class HELMMultiViewRouter(nn.Module):
    """Minimal sequence-level elastic router for HELM_7e.

    There are 8 permanent heads and 24 elastic candidates. The elastic router uses
    ordinary learned logits z_h(x). Forward routing is hard: z_h > 0. The same hard
    mask is wrapped in a sigmoid STE so CE and the count loss can train the router.

    Easiness labels are training-time supervision only. They are converted to a
    desired total head count in [8, 32]. At inference no easiness value is needed.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads

        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias=config.bias,
        )
        self.l_i_weights = nn.Parameter(torch.ones(config.num_router_latents))

        # IMPORTANT: unlike Phase 13, q_up_proj is NOT normalized. Magnitude is allowed
        # to carry information. We monitor its norms instead of pre-emptively constraining it.
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            self.num_elastic_candidates,
            bias=False,
        )

        # Last-forward telemetry.
        self.save_router_logits = None
        self.save_sigmoid_scores = None
        self.save_hard_mask = None
        self.save_total_head_count = None
        self.save_target_total_head_count = None
        self.save_count_error = None
        self.save_count_loss = None

    def _easiness_to_target(self, easiness_score):
        """Map easiness label -> integer target total heads in [8, 32].

        Easiness is converted to a CDF quantile q so the target depends on relative
        difficulty rather than the raw label's dataset-specific numeric scale:
          q=0   (hardest) -> 32 total heads
          q=0.5 (median)  -> 16 total heads
          q=1   (easiest) -> 8 total heads
        """
        batch = easiness_score.numel()
        device = easiness_score.device
        e = easiness_score.to(torch.float32).reshape(batch).clamp(0.0, 1.0)

        bp = self.config.easiness_cdf_breakpoints
        if bp is not None and len(bp) >= 2:
            breaks = torch.as_tensor(bp, device=device, dtype=torch.float32)
            n_intervals = breaks.numel() - 1
            pos = torch.searchsorted(breaks, e, right=True).clamp(1, n_intervals)
            lo = breaks[pos - 1]
            hi = breaks[pos]
            frac = (e - lo) / (hi - lo + 1e-8)
            q = ((pos - 1).to(torch.float32) + frac) / float(n_intervals)
            q = q.clamp(0.0, 1.0)
        else:
            # Safe fallback if no breakpoint table was supplied.
            q = e

        h_min = float(self.config.head_target_min)
        h_ctr = float(self.config.head_target_center)
        h_max = float(self.config.head_target_max)

        hard_half = q < 0.5
        hard_target = h_ctr + (h_max - h_ctr) * ((0.5 - q) / 0.5)
        easy_target = h_ctr + (h_min - h_ctr) * ((q - 0.5) / 0.5)
        target_total = torch.where(hard_half, hard_target, easy_target)

        # Actual executed counts are integer, so make an exactly attainable target.
        return target_total.round().clamp(h_min, h_max)

    def forward(self, hidden_states, easiness_score=None):
        # ----- Existing HELM multi-latent sequence summary -----
        q_down = justnorm(self.q_down_proj.weight, dim=1).to(hidden_states.dtype)
        scanner = F.linear(hidden_states, q_down)                       # [B,S,R]
        scanner_weights = F.softmax(self.scale * scanner, dim=1)       # [B,S,R]
        latents = torch.bmm(scanner_weights.transpose(1, 2), hidden_states)  # [B,R,D]

        latent_weights = F.softmax(self.l_i_weights, dim=0)
        pooled = (latents * latent_weights.view(1, -1, 1)).sum(dim=1)  # [B,D]

        # ----- Minimal learned router -----
        router_logits = cast_linear(pooled, self.q_up_proj)                 # [B,E]
        sigmoid_scores = torch.sigmoid(router_logits)
        hard_mask = (router_logits > 0).to(router_logits.dtype)

        # Forward = exact 0/1 hard mask. Backward = sigmoid derivative.
        ste_mask = hard_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        actual_elastic_count = hard_mask.sum(dim=-1)
        actual_total_count = actual_elastic_count + float(self.config.num_permanent_heads)

        # ----- Easiness-supervised ACTUAL hard-count loss -----
        if easiness_score is not None:
            target_total_count = self._easiness_to_target(easiness_score)
            target_elastic_count = target_total_count - float(self.config.num_permanent_heads)

            # ste_mask has the hard count as its forward value but keeps a sigmoid
            # backward path. This avoids the old sum(sigmoid) soft-count loophole.
            differentiable_elastic_count = ste_mask.float().sum(dim=-1)
            count_error = differentiable_elastic_count - target_elastic_count.float()
            denom = float(self.num_elastic_candidates)
            count_loss = (
                float(self.config.count_loss_lambda)
                * (count_error / denom).square().mean()
            )
        else:
            if self.training:
                raise ValueError("HELM_7e training requires easiness_score")
            target_total_count = torch.full_like(actual_total_count, -1.0)
            count_error = torch.zeros_like(actual_total_count)
            count_loss = router_logits.new_zeros(())

        # ----- Telemetry -----
        self.save_router_logits = router_logits.detach()
        self.save_sigmoid_scores = sigmoid_scores.detach()
        self.save_hard_mask = hard_mask.detach()
        self.save_total_head_count = actual_total_count.detach()
        self.save_target_total_head_count = target_total_count.detach()
        self.save_count_error = (actual_total_count - target_total_count).detach()
        self.count_loss = count_loss
        self.save_count_loss = count_loss.detach()

        router_mask = ste_mask.view(ste_mask.size(0), -1, 1, 1)
        if self.config.num_permanent_heads > 0:
            permanent = torch.ones(
                ste_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype,
            )
            router_mask = torch.cat((permanent, router_mask), dim=1)

        return router_mask


class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config, layer_idx=0):
        super().__init__()

        self.layer_idx = int(layer_idx)
        self.rank_ratio_floor = (
            float(config.rank_floor_scale) * float(config.rank_reference_ratios[self.layer_idx])
        )
        self.save_rank_value = None
        self.save_rank_ratio = None
        self.save_rank_floor = float(self.rank_ratio_floor)
        self.save_rank_penalty = None

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.d_head = config.d_head if config.d_head is not None else (config.hidden_size // config.num_attention_heads)
        self.total_head_dim = self.num_attention_heads * self.d_head   
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.total_head_dim))  # was: self.hidden_size

        # Output Matrix
        self.output = nn.Linear(
            self.total_head_dim,      # was: config.hidden_size
            config.hidden_size,
            bias = config.bias
        )


    def _compute_rank_regularizer(self, context_layer, router_mask):
        """Direct raw spectral participation rank of the EXECUTED head contexts.

        This is a training-only auxiliary objective.

        Let X_h be the routed context contribution of head h, flattened over a
        fixed-size token sample and d_head. For each example:
            G = X X^T
            r_PR = tr(G)^2 / tr(G^2)

        r_PR is a raw effective-rank measure: it falls when energy/covariance is
        concentrated into a small number of head-space directions.

        We normalize by the ACTUAL hard-forward head count K so the objective asks
        for a healthy fraction of the currently allocated compute, rather than
        simply rewarding more active heads.

        Crucially, X uses the normal HELM_7e hard/STE router mask. An OFF head is
        numerically absent, and its own Q/K/V path receives no hidden "ghost"
        training like HELM_7d.
        """
        B, H, S, D = context_layer.shape

        # Keep the rank calculation's shape small and nearly constant across the
        # 1024/2048/4096 curriculum. S is static inside each compiled XLA graph.
        sample_tokens = max(1, int(self.config.rank_sample_tokens))
        stride = max(1, S // sample_tokens)
        sampled = context_layer[:, :, ::stride, :]
        if sampled.size(2) > sample_tokens:
            sampled = sampled[:, :, :sample_tokens, :]

        # Rank loss may train the active head representations,
        # but it must not directly train router selection.
        rank_mask = router_mask.detach()
        
        routed = sampled * rank_mask.expand_as(sampled)

        # [B,H,T,Dh] -> [B,H,N]. Normalize N only for numerical scale; effective
        # rank itself is scale invariant.
        flat = routed.float().reshape(B, H, -1)
        flat = flat / math.sqrt(float(max(1, flat.size(-1))))

        # Batched [B,H,H] Gram matrix; no eigendecomposition is required.
        gram = torch.bmm(flat, flat.transpose(1, 2))
        trace = torch.diagonal(gram, dim1=-2, dim2=-1).sum(dim=-1)
        trace_g2 = gram.square().sum(dim=(-2, -1))
        rank_value = trace.square() / (trace_g2 + 1e-12)

        # K uses detached hard-forward values. Dividing by K prevents the rank
        # regularizer from "winning" merely by switching on more heads.
        hard_count = rank_mask[:, :, 0, 0].float().sum(dim=-1).clamp_min(1.0)
        rank_ratio = rank_value / hard_count

        floor = float(self.rank_ratio_floor)
        deficit = F.relu(rank_ratio.new_tensor(floor) - rank_ratio)
        penalty = deficit.square().mean()

        self.save_rank_value = rank_value.detach()
        self.save_rank_ratio = rank_ratio.detach()
        self.save_rank_floor = float(floor)
        self.save_rank_penalty = penalty.detach()

        return penalty

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        # Obtain Hidden Size
        batch_size, seq_len, _ = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)


        eval_backend = self._eval_backend

        # Reshape q,k,v
        # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

        # Reshape q,k,v
        # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)


        # TRAINING / TPU MODE
        if (self.training or eval_backend == "dense"):

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Apply Attention
            # Scale by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head),
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Training-only rank objective. It is evaluated BEFORE permanent-head
            # jitter so the auxiliary target does not chase random dropout noise.
            if self.training and router_mask is not None:
                rank_penalty = self._compute_rank_regularizer(context_layer, router_mask)
            else:
                rank_penalty = context_layer.new_zeros(())
                self.save_rank_value = None
                self.save_rank_ratio = None
                self.save_rank_penalty = rank_penalty.detach()

            self.rank_penalty = rank_penalty

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # FLEX ATTENTION (for GPUs)
        elif eval_backend == "flex" and batch_size > 1:

             # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Router_mask scores, 1 or 0 or sigmoid scaling
            # [b, num_attention_heads, 1 , 1] -> [batch, num_attention_heads]
            active = (router_mask[:, :, 0, 0] > 0)

            # Boolean attention mask
            # [batch_size, 1, 1, seq_len] -> [batch, seq_len]
            key_valid = (attention_mask[:, 0, 0, :] >=0)

            # mask_mod: attend / calculate only if the head is on and its not a padding token
            def mask_mod(bi, hi, qi, ki):
                return active[bi, hi] & key_valid[bi, ki]

            # Prep the block to be passed into flex attention
            block_mask = self._build_block_mask(
                mask_mod, batch_size, self.num_attention_heads,seq_len, q.device
            )

            # Apply flex attention
            context_layer = self._flex_attn(
                q, k, v, block_mask = block_mask, scale = math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Apply router mask to 0 the heads of the context layer
            # [batch, num attention heads, seq_len, head dim] (router_mask [batch, num_attention_heads, 1,1] was broadcasted)
            context_layer = context_layer * router_mask.expand_as(context_layer)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # Single query effieincy
        else:

            # This path only looks at batch element 0's router decisions (see below), so
            # it is only correct for batch_size == 1 -- each example's active heads are
            # data-dependent, so silently reusing example 0's mask for other examples
            # would produce wrong outputs for them instead of a loud failure.
            assert batch_size == 1, (
                f"HELMSelfAttention's 'gather' eval backend only supports batch_size == 1 "
                f"(got batch_size={batch_size}); use backend='flex' for batched inference."
            )

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # q, k, v are already [b, num_attention_heads, seq_len, d_head] from the
            # shared reshape/permute above -- no need to reshape them again here.

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor to q and k
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced

            # Flash Attention (only for GPUs where on-the-fly splicing can exist)
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims].to(context_reshaped.dtype)
            sliced_bias = None if self.output.bias is None else self.output.bias.to(context_reshaped.dtype)

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=sliced_bias)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention + HELMMLP
class HELMBlock(nn.Module):

    def __init__(self, config, layer_idx):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config, layer_idx=layer_idx)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask, easiness_score):
        router_mask = self.mlt_vw_rtr(hidden_states, easiness_score)
        count_loss = self.mlt_vw_rtr.count_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        rank_penalty = self.attn.rank_penalty
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, count_loss, rank_penalty


class HELMModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList([HELMBlock(config, layer_idx=i) for i in range(config.num_hidden_layers)])

    def forward(self, input_ids, attention_mask, easiness_score=None):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)
        total_count_loss = hidden_states.new_zeros(())
        total_rank_penalty = hidden_states.new_zeros(())

        for block in self.blocks:
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, count_loss, rank_penalty = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states, count_loss, rank_penalty = block(hidden_states, attention_mask, easiness_score)
            total_count_loss = total_count_loss + count_loss
            total_rank_penalty = total_rank_penalty + rank_penalty

        # Both auxiliary objectives are per-layer averages, so their coefficients
        # remain independent of model depth.
        total_count_loss = total_count_loss / float(len(self.blocks))
        total_rank_penalty = total_rank_penalty / float(len(self.blocks))
        return hidden_states, total_count_loss, total_rank_penalty


class HELMForMaskedLM(PreTrainedModel):

    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.vocab_size, bias=config.bias)
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # q_up_proj is intentionally EXCLUDED: HELM_7e allows router magnitude.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}

        for i, block in enumerate(self.model.blocks):
            router = block.mlt_vw_rtr
            logits = router.save_router_logits.float().cpu()
            sigmoid = router.save_sigmoid_scores.float().cpu()
            hard = router.save_hard_mask.float().cpu()
            actual = router.save_total_head_count.float().cpu()
            target = router.save_target_total_head_count.float().cpu()
            error = router.save_count_error.float().cpu()
            q_up_norms = router.q_up_proj.weight.detach().float().norm(dim=1).cpu()
            attn = block.attn

            telemetry[f"layer_{i}_router_logits"] = logits
            telemetry[f"layer_{i}_sigmoid_scores"] = sigmoid
            telemetry[f"layer_{i}_hard_mask"] = hard
            telemetry[f"layer_{i}_elastic_head_ratio"] = hard.mean().item()
            telemetry[f"layer_{i}_total_head_count_mean"] = actual.mean().item()
            telemetry[f"layer_{i}_target_head_count_mean"] = target.mean().item()
            telemetry[f"layer_{i}_count_error_mean"] = error.mean().item()
            telemetry[f"layer_{i}_count_error_mae"] = error.abs().mean().item()
            telemetry[f"layer_{i}_count_loss"] = router.save_count_loss.float().item()
            telemetry[f"layer_{i}_router_weight_norms"] = q_up_norms
            telemetry[f"layer_{i}_router_weight_norm_mean"] = q_up_norms.mean().item()
            telemetry[f"layer_{i}_router_weight_norm_std"] = q_up_norms.std(unbiased=False).item()
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().float().cpu()

            if attn.save_rank_ratio is not None:
                rr = attn.save_rank_ratio.detach().float().cpu()
                rv = attn.save_rank_value.detach().float().cpu()
                telemetry[f"layer_{i}_rank_ratio_mean"] = rr.mean().item()
                telemetry[f"layer_{i}_rank_ratio_min"] = rr.min().item()
                telemetry[f"layer_{i}_rank_value_mean"] = rv.mean().item()
                telemetry[f"layer_{i}_rank_floor"] = float(attn.save_rank_floor)
                telemetry[f"layer_{i}_rank_penalty"] = attn.save_rank_penalty.float().item()

        return telemetry

    def forward(self, input_ids, attention_mask, current_step=None, easiness_score=None):
        # current_step is accepted only for backward compatibility with older callers.
        features, total_count_loss, total_rank_penalty = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            easiness_score=easiness_score,
        )

        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)
        unscaled_logits = cast_linear(features, self.classifier)
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits
        return logits, total_count_loss, total_rank_penalty


Writing model_7e.py


In [10]:
%%writefile model_dense32.py
"""HELM Dense-32 baseline.

Clean dense control for HELM_7c:
- 12 layers
- d_model = 1024
- 32 attention heads
- d_head = 64
- total attention width = 2048
- all 32 heads active for every example in every layer
- no router, no permanent/elastic split, no easiness supervision,
  no count loss, and no router-specific jitter.

The nGPT attention/MLP mechanics are intentionally kept aligned with HELM_7c.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None

from transformers import PretrainedConfig, PreTrainedModel


def justnorm(x, dim=-1, eps=1e-12):
    return x / (x.norm(p=2, dim=dim, keepdim=True) + eps)


def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x, w, b)


class HELMConfig(PretrainedConfig):
    model_type = "helm_dense32"

    def __init__(
        self,
        # General model hyperparameters
        hidden_size=1024,
        sqrt_hidden_size=32,
        max_position_embeddings=4096,
        initializer_range=0.03125,
        num_hidden_layers=12,
        num_attention_heads=32,
        d_head=64,
        rope_theta=160000,
        intermediate_size=2816,
        norm_eps=1e-12,
        hidden_act="swiglu",
        swiglu_s_init=1.0,
        base_lr=3e-4,
        min_lr=3e-5,
        weight_decay=0.0,
        bias=False,
        use_ckpt=False,

        # Tokenization / MLM metadata
        tokenizer_path="answerdotai/ModernBERT-base",
        vocab_size=50368,
        bos_token_id=50281,
        eos_token_id=50282,
        pad_token_id=50283,
        mask_token_id=50284,
        unk_token_id=50285,
        mlm_probability=0.3,
        mlm_use_span_masking=True,
        mlm_span_length=3,

        # nGPT attention / FFN hyperparameters
        ngpt_sqk_init_value=1.0,
        ngpt_sqk_init_scale=0.03125,
        use_exclusive_attention=True,
        ngpt_alpha_value_attn=0.05,
        ngpt_alpha_scale_attn=0.03125,
        ngpt_alpha_value_mlp=0.05,
        ngpt_alpha_scale_mlp=0.03125,
        ngpt_suv_value=1.0,
        ngpt_suv_scale=1.0,
        ngpt_sz_init_value=1.0,
        ngpt_sz_init_scale=0.03125,
        **kwargs,
    ):
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale

        if self.num_attention_heads != 32:
            raise ValueError(
                f"Dense-32 control requires num_attention_heads=32, got {self.num_attention_heads}."
            )
        if self.d_head != 64:
            raise ValueError(f"Dense-32 control requires d_head=64, got {self.d_head}.")
        if self.num_attention_heads * self.d_head != 2048:
            raise ValueError("Dense-32 attention width must be exactly 2048.")

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id,
        )

    def forward(self, input_ids):
        return justnorm(self.word_embeddings(input_ids))


class RotaryEmbeddings(nn.Module):
    def __init__(self, dim, max_position_embeddings, rope_theta=160000):
        super().__init__()
        inv_freq = 1.0 / (
            rope_theta ** (torch.arange(0, dim, 2).float() / dim)
        )
        t = torch.arange(max_position_embeddings, dtype=inv_freq.dtype)
        freqs = torch.outer(t, inv_freq)
        freqs = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x):
        seq_len = x.shape[-2]
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)
        return (x * x_cos) + (self.rotate_half(x) * x_sin)


class HELMSelfAttention(nn.Module):
    """Dense 32-head attention. Every head is always active."""

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.d_head = config.d_head
        self.total_head_dim = self.num_attention_heads * self.d_head
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        # 1024 -> Q/K/V, each 2048 wide.
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias=config.bias,
        )

        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta,
        )

        self.sqk = nn.Parameter(
            self.ngpt_sqk_init_scale * torch.ones(self.total_head_dim)
        )

        # 2048 -> 1024, matching HELM_7c.
        self.output = nn.Linear(
            self.total_head_dim,
            config.hidden_size,
            bias=config.bias,
        )

    def forward(self, hidden_states, attention_mask):
        batch_size, seq_len, _ = hidden_states.shape

        qkv_proj = cast_linear(hidden_states, self.qkv)
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        q = q.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)
        k = k.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)
        v = v.view(
            batch_size, seq_len, self.num_attention_heads, self.d_head
        ).permute(0, 2, 1, 3)

        # Keep HELM/nGPT mechanics unchanged.
        q = justnorm(q)
        k = justnorm(k)
        q = self.RoPE(q)
        k = self.RoPE(k)

        sqk = self.sqk * (
            self.ngpt_sqk_init_value / self.ngpt_sqk_init_scale
        )
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)
        q = sqk.to(q.dtype) * q
        k = sqk.to(k.dtype) * k

        context_layer = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask.to(q.dtype),
            scale=math.sqrt(self.d_head),
        )

        if self.config.use_exclusive_attention:
            vn = F.normalize(v, dim=-1)
            context_layer = (
                context_layer
                - (context_layer * vn).sum(dim=-1, keepdim=True) * vn
            )

        context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()
        context_reshaped = context_reshaped.view(batch_size, seq_len, self.total_head_dim)
        return cast_linear(context_reshaped, self.output)


class HELMMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        self.attn_alpha = nn.Parameter(
            self.ngpt_alpha_scale_attn * torch.ones(self.hidden_size)
        )
        self.mlp_alpha = nn.Parameter(
            self.ngpt_alpha_scale_mlp * torch.ones(self.hidden_size)
        )

        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias=config.bias,
        )
        self.suv = nn.Parameter(
            self.ngpt_suv_scale * torch.ones(2 * self.intermediate_size)
        )
        self.silu = nn.SiLU()
        self.mlp_proj = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias,
        )

    def forward(self, hidden_states, hidden_states_attention):
        # nGPT attention residual update.
        a_norm = justnorm(hidden_states)
        b_norm = justnorm(hidden_states_attention)
        lr = self.attn_alpha * (
            self.ngpt_alpha_value_attn / self.ngpt_alpha_scale_attn
        )
        lr = torch.abs(lr).to(a_norm.dtype)
        hidden_states_opt1 = justnorm(a_norm + lr * (b_norm - a_norm))

        # nGPT SwiGLU FFN.
        uv_pre = cast_linear(hidden_states_opt1, self.mlp_exp)
        suv = self.suv * (
            self.ngpt_suv_value / self.ngpt_suv_scale
        ) * (self.hidden_size ** 0.5)
        uv_post_suv = suv.to(uv_pre.dtype) * uv_pre
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)
        x_mlp = u * self.silu(v)
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # nGPT MLP residual update.
        a_norm = justnorm(hidden_states_opt1)
        b_norm = justnorm(h_mlp)
        lr = self.mlp_alpha * (
            self.ngpt_alpha_value_mlp / self.ngpt_alpha_scale_mlp
        )
        lr = torch.abs(lr).to(a_norm.dtype)
        return justnorm(a_norm + lr * (b_norm - a_norm))


class HELMBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask):
        attn_output = self.attn(hidden_states, attention_mask)
        return self.mlp(hidden_states, attn_output)


class HELMModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList(
            [HELMBlock(config) for _ in range(config.num_hidden_layers)]
        )

    def forward(self, input_ids, attention_mask):
        # Additive SDPA mask: [B,S] -> [B,1,1,S].
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(
            attention_mask == 0, float("-inf")
        )
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)

        for block in self.blocks:
            if self.use_ckpt and self.training:
                ckpt_fn = (
                    _xla_checkpoint
                    if (
                        _xla_checkpoint is not None
                        and hidden_states.device.type == "xla"
                    )
                    else torch.utils.checkpoint.checkpoint
                )
                hidden_states = ckpt_fn(
                    block,
                    hidden_states,
                    attention_mask,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states = block(hidden_states, attention_mask)

        return hidden_states


class HELMForMaskedLM(PreTrainedModel):
    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=config.bias,
        )
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.initializer_range,
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.config.initializer_range,
            )

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # Every matrix here existed in the dense-16 baseline as well.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}
        for i, block in enumerate(self.model.blocks):
            attn = block.attn
            mlp = block.mlp

            sqk = attn.sqk.detach().float().cpu()
            telemetry[f"layer_{i}_sqk_mean"] = sqk.mean().item()
            telemetry[f"layer_{i}_sqk_std"] = sqk.std(unbiased=False).item()
            telemetry[f"layer_{i}_sqk_hist"] = sqk

            attn_alpha = mlp.attn_alpha.detach().float().cpu()
            telemetry[f"layer_{i}_attn_alpha_mean"] = attn_alpha.mean().item()
            telemetry[f"layer_{i}_attn_alpha_std"] = attn_alpha.std(unbiased=False).item()
            telemetry[f"layer_{i}_attn_alpha_hist"] = attn_alpha

            mlp_alpha = mlp.mlp_alpha.detach().float().cpu()
            telemetry[f"layer_{i}_mlp_alpha_mean"] = mlp_alpha.mean().item()
            telemetry[f"layer_{i}_mlp_alpha_std"] = mlp_alpha.std(unbiased=False).item()
            telemetry[f"layer_{i}_mlp_alpha_hist"] = mlp_alpha

            suv = mlp.suv.detach().float().cpu()
            telemetry[f"layer_{i}_suv_mean"] = suv.mean().item()
            telemetry[f"layer_{i}_suv_std"] = suv.std(unbiased=False).item()
            telemetry[f"layer_{i}_suv_hist"] = suv

        sz = self.sz.detach().float().cpu()
        telemetry["lm_head_sz_mean"] = sz.mean().item()
        telemetry["lm_head_sz_std"] = sz.std(unbiased=False).item()
        telemetry["lm_head_sz_hist"] = sz
        return telemetry

    def forward(
        self,
        input_ids,
        attention_mask,
        current_step=None,
        easiness_score=None,
    ):
        # current_step/easiness_score are accepted only so old generic callers do
        # not break. They have ZERO effect on this dense model.
        features = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        sz = self.sz * (
            self.ngpt_sz_init_value / self.ngpt_sz_init_scale
        )
        unscaled_logits = cast_linear(features, self.classifier)
        return sz.to(unscaled_logits.dtype) * unscaled_logits

Writing model_dense32.py


In [12]:
%%writefile analyze_7e_vs_dense32_attention_anatomy.py
"""
HELM_7e vs Dense-32 attention anatomy audit.

Purpose
-------
Do NOT ask only "how many effective head directions exist?"
Instead decompose each attention head into:

1. QK / READ behavior
   - pre-softmax QK logit statistics
   - normalized attention entropy
   - effective number of attended tokens
   - max attention probability
   - top-5 attention mass
   - expected attention distance
   - self-attention mass
   - low-rank QK operator spectrum

2. OV / WRITE behavior
   - V RMS
   - context RMS
   - residual-write RMS after W_O^(h)
   - attention mixing gain: context_power / value_power
   - W_O block Frobenius norm
   - context->residual write gain
   - alignment gain between context covariance and W_O strong directions
   - low-rank OV operator spectrum

3. Magnitude origin
   For head h:
       Y_h = C_h W_O^(h)

   We decompose its total residual energy as:

       E_total(Y_h)
         = context_power
         * ||W_O^(h)||_F^2
         * alignment_gain

   where alignment_gain = 1 under an isotropic context second moment.

4. Optional CURRENT gradient probe
   Without retraining, run one fixed MLM batch and inspect per-head
   ||grad W_Q||, ||grad W_K||, ||grad W_V||, ||grad W_O||.

   This is NOT cumulative optimization history. It is the local gradient field at
   the checkpoint. True historical gradients cannot be reconstructed unless they
   were logged during training, though Adam moments may exist in optimizer state.

Comparison
----------
The script evaluates HELM_7e and native Dense-32 on the SAME deterministic
validation examples and same MLM masking.

For HELM_7e:
- hidden states come from its normal routed forward pass
- anatomy reconstructs ALL 32 candidate heads at the analyzed layer before the
  current layer's routing mask, so weak/off candidate heads remain inspectable
- activation frequency is attached to each head as another variable

For Dense-32:
- all 32 heads are naturally active

IMPORTANT DATATYPE RULE
-----------------------
Any tensor converted to NumPy uses:

    tensor.detach().to(torch.float32).cpu().numpy().astype(np.float64)

Example
-------
python analyze_7e_vs_dense32_attention_anatomy.py \
    --device xla \
    --model-7e model_7e.py \
    --model-dense model_dense32.py \
    --checkpoint-7e-path /path/to/checkpoint-006500.pt \
    --checkpoint-dense-path /path/to/checkpoint-006500.pt \
    --num-examples 16 \
    --batch-size 2 \
    --layers 0,1,2,3,4,5,6,7,8,9,10,11 \
    --qk-query-tokens 64 \
    --gradient-probe
"""

from pathlib import Path
import argparse
import contextlib
import csv
import json
import math
import shutil

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from helm_analysis_core import *


DEFAULT_7E_REPO = "JamesResearch1216/HELM_7e"
DEFAULT_DENSE_REPO = "JamesResearch1216/HELM_Vanilla_32"


# ---------------------------------------------------------------------------
# Small linear-algebra helpers
# ---------------------------------------------------------------------------

def np64(t):
    return (
        t.detach()
        .to(torch.float32)
        .cpu()
        .numpy()
        .astype(np.float64)
    )


def psd_sqrt_np(mat):
    mat = 0.5 * (mat + mat.T)
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)[None, :]) @ vecs.T


def low_rank_product_singular_values(A_torch, B_torch):
    """
    Singular values of A @ B without forming the large D x D product.

    A: [D, r]
    B: [r, D]

    Nonzero squared singular values of AB are eigenvalues of:
        sqrt(A^T A) (B B^T) sqrt(A^T A)

    which is only r x r (r=d_head=64 here).
    """
    A = np64(A_torch)
    B = np64(B_torch)

    ata = A.T @ A
    bbt = B @ B.T

    s_ata = psd_sqrt_np(ata)
    small = s_ata @ bbt @ s_ata
    small = 0.5 * (small + small.T)

    eig = np.linalg.eigvalsh(small)
    eig = np.clip(eig, 0.0, None)
    singular = np.sqrt(eig)
    return np.sort(singular)[::-1]


def spectrum_metrics(singular):
    s = np.asarray(singular, dtype=np.float64)
    s = s[s > 1e-14]

    if len(s) == 0:
        return {
            "spectral_norm": 0.0,
            "frobenius_norm": 0.0,
            "stable_rank": 0.0,
            "singular_entropy_rank": 0.0,
            "singular_participation_rank": 0.0,
        }

    spectral = float(s.max())
    frob = float(np.sqrt(np.square(s).sum()))
    stable = float((frob * frob) / (spectral * spectral + 1e-18))

    p = s / s.sum()
    entropy_rank = float(np.exp(-(p * np.log(p + 1e-18)).sum()))
    participation = float(
        (s.sum() ** 2) / (np.square(s).sum() + 1e-18)
    )

    return {
        "spectral_norm": spectral,
        "frobenius_norm": frob,
        "stable_rank": stable,
        "singular_entropy_rank": entropy_rank,
        "singular_participation_rank": participation,
    }


def qkv_weight_blocks(attn):
    H = int(attn.num_attention_heads)
    d = int(attn.d_head)
    D = int(attn.hidden_size)

    w = attn.qkv.weight
    q, k, v = w.split(attn.total_head_dim, dim=0)

    q = q.view(H, d, D)
    k = k.view(H, d, D)
    v = v.view(H, d, D)
    return q, k, v


def output_weight_blocks(attn):
    # output.weight is [D, H*d]
    D = int(attn.hidden_size)
    H = int(attn.num_attention_heads)
    d = int(attn.d_head)
    return attn.output.weight.view(D, H, d).permute(1, 0, 2)  # [H,D,d]


# ---------------------------------------------------------------------------
# Capture layer inputs and reconstruct q/k/v/context
# ---------------------------------------------------------------------------

class LayerInputCapture:
    def __init__(self, model, layers):
        self.model = model
        self.layers = list(layers)
        self.data = {}
        self.handles = []

    def _hook(self, li):
        def hook(module, inputs):
            self.data[li] = (
                inputs[0].detach(),
                inputs[1].detach(),
            )
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(
                    self._hook(li)
                )
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def reconstruct_qkv_context(module, attn, hidden_states, attention_mask):
    """
    Match the model's actual attention mechanics exactly.
    Returns q,k,v,context with shapes [B,H,S,d].
    """
    qkv_proj = module.cast_linear(hidden_states, attn.qkv)
    B, S, _ = hidden_states.shape

    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)
    q = q.view(B, S, attn.num_attention_heads, attn.d_head).permute(0,2,1,3)
    k = k.view(B, S, attn.num_attention_heads, attn.d_head).permute(0,2,1,3)
    v = v.view(B, S, attn.num_attention_heads, attn.d_head).permute(0,2,1,3)

    q = module.justnorm(q)
    k = module.justnorm(k)

    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = attn.sqk * (
        attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale
    )
    sqk = sqk.view(
        1, attn.num_attention_heads, 1, attn.d_head
    ).to(q.dtype)

    q = sqk * q
    k = sqk * k

    context = F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )

    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (
            context * vn
        ).sum(dim=-1, keepdim=True) * vn

    return q, k, v, context


# ---------------------------------------------------------------------------
# QK functional statistics
# ---------------------------------------------------------------------------

def qk_functional_metrics(q, k, attention_mask, query_tokens):
    """
    q,k: [B,H,S,d]
    attention_mask: [B,1,1,S] additive 0/-inf

    Use sampled QUERY positions but all KEY positions.
    """
    B, H, S, d = q.shape
    T = min(int(query_tokens), S)

    qpos = torch.linspace(
        0, S - 1, steps=T, device=q.device
    ).long()
    qs = q.index_select(2, qpos)  # [B,H,T,d]

    logits = torch.matmul(
        qs.to(torch.float32),
        k.to(torch.float32).transpose(-2, -1),
    ) * math.sqrt(d)  # exactly match model's explicit SDPA scale

    additive_mask = attention_mask.to(torch.float32)
    logits = logits + additive_mask

    probs = torch.softmax(logits, dim=-1)
    logp = torch.log(probs.clamp_min(1e-30))

    entropy = -(probs * logp).sum(dim=-1)  # [B,H,T]

    key_valid = (attention_mask[:, 0, 0, :] >= 0)
    n_valid = key_valid.sum(dim=-1).to(torch.float32).clamp_min(2.0)  # [B]
    normalized_entropy = entropy / torch.log(
        n_valid.view(B,1,1)
    )

    effective_tokens = torch.exp(entropy)
    max_prob = probs.max(dim=-1).values

    k5 = min(5, S)
    top5_mass = probs.topk(k=k5, dim=-1).values.sum(dim=-1)

    key_positions = torch.arange(
        S, device=q.device, dtype=torch.float32
    )
    q_positions = qpos.to(torch.float32)
    dist = (
        q_positions.view(T,1) - key_positions.view(1,S)
    ).abs() / max(1.0, float(S - 1))
    expected_distance = (
        probs * dist.view(1,1,T,S)
    ).sum(dim=-1)

    # Probability assigned exactly to query's own position.
    gather_idx = qpos.view(1,1,T,1).expand(B,H,T,1)
    self_mass = probs.gather(-1, gather_idx).squeeze(-1)

    # Raw finite QK logits before mask for scale/contrast.
    raw_logits = torch.matmul(
        qs.to(torch.float32),
        k.to(torch.float32).transpose(-2, -1),
    ) * math.sqrt(d)

    query_valid = key_valid.index_select(1, qpos)  # [B,T]
    qmask = query_valid.view(B,1,T).to(torch.float32)

    def masked_head_mean(x):
        # x [B,H,T]
        denom = qmask.sum(dim=(0,2)).clamp_min(1.0)  # [1] broadcast H
        return (x * qmask).sum(dim=(0,2)) / denom

    entropy_h = masked_head_mean(entropy)
    norm_entropy_h = masked_head_mean(normalized_entropy)
    eff_h = masked_head_mean(effective_tokens)
    max_h = masked_head_mean(max_prob)
    top5_h = masked_head_mean(top5_mass)
    dist_h = masked_head_mean(expected_distance)
    self_h = masked_head_mean(self_mass)

    # For logits, restrict keys to valid positions by NaN masking then use
    # per-head aggregate moments.
    key_mask = key_valid.view(B,1,1,S)
    raw = raw_logits.masked_fill(~key_mask, float("nan"))

    logit_mean = torch.nanmean(raw, dim=(0,2,3))
    # Manual nan std.
    centered = raw - logit_mean.view(1,H,1,1)
    sq = centered.square()
    valid_count = torch.isfinite(sq).sum(dim=(0,2,3)).clamp_min(1)
    logit_std = torch.sqrt(
        torch.nansum(sq, dim=(0,2,3)) / valid_count
    )

    return {
        "attention_entropy": entropy_h,
        "normalized_attention_entropy": norm_entropy_h,
        "effective_attended_tokens": eff_h,
        "max_attention_probability": max_h,
        "top5_attention_mass": top5_h,
        "expected_attention_distance": dist_h,
        "self_attention_mass": self_h,
        "qk_logit_mean": logit_mean,
        "qk_logit_std": logit_std,
    }


# ---------------------------------------------------------------------------
# OV/magnitude functional statistics
# ---------------------------------------------------------------------------

def ov_magnitude_metrics(v, context, output_blocks):
    """
    v/context [B,H,S,d]
    output_blocks [H,D,d]

    Returns per-head torch tensors plus alignment gain.
    """
    B, H, S, d = context.shape
    D = output_blocks.size(1)

    v32 = v.to(torch.float32)
    c32 = context.to(torch.float32)
    W = output_blocks.to(torch.float32)

    value_power = v32.square().mean(dim=(0,2,3))       # [H]
    context_power = c32.square().mean(dim=(0,2,3))     # [H]

    # y[b,h,s,o] = c[b,h,s,d] @ W[h,o,d]^T
    y = torch.einsum("bhsd,hod->bhso", c32, W)
    residual_power = y.square().mean(dim=(0,2,3))      # mean over B,S,D

    value_rms = torch.sqrt(value_power.clamp_min(0))
    context_rms = torch.sqrt(context_power.clamp_min(0))
    residual_rms = torch.sqrt(residual_power.clamp_min(0))

    w_frob_sq = W.square().sum(dim=(1,2))
    w_frob = torch.sqrt(w_frob_sq)

    mixing_gain = context_power / (value_power + 1e-18)

    # Total residual energy per token = D * residual_power.
    residual_total_energy = float(D) * residual_power

    # If C second moment were isotropic:
    # M = context_power * I_d
    # E||Y||^2 = context_power * ||W||_F^2
    isotropic_expected = context_power * w_frob_sq
    alignment_gain = residual_total_energy / (
        isotropic_expected + 1e-18
    )

    # context -> residual RMS gain.
    write_rms_gain = residual_rms / (context_rms + 1e-18)

    return {
        "value_rms": value_rms,
        "context_rms": context_rms,
        "residual_rms": residual_rms,
        "value_power": value_power,
        "context_power": context_power,
        "residual_power": residual_power,
        "attention_mixing_gain_power": mixing_gain,
        "output_block_frobenius": w_frob,
        "output_block_frobenius_sq": w_frob_sq,
        "context_to_residual_rms_gain": write_rms_gain,
        "alignment_gain": alignment_gain,
    }


# ---------------------------------------------------------------------------
# Per-model anatomy
# ---------------------------------------------------------------------------

def capture_activation_frequency(model, batches, dev):
    """7e only. Dense-32 returns ones."""
    H = int(model.config.num_attention_heads)
    L = len(model.model.blocks)

    if not hasattr(model.model.blocks[0], "mlt_vw_rtr"):
        return np.ones((L,H), dtype=np.float64)

    counts = np.zeros((L,H), dtype=np.float64)
    n = 0
    P = int(model.config.num_permanent_heads)

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                _ = call_model(model, batch, pass_easiness=True)
            dev.mark_step()

            B = cpu_batch["input_ids"].size(0)
            n += B

            for li, block in enumerate(model.model.blocks):
                elastic = np64(
                    block.mlt_vw_rtr.save_hard_mask
                )  # [B,E]
                counts[li, :P] += B
                counts[li, P:] += elastic.sum(axis=0)

    return counts / max(1,n)


def analyze_model(
    label,
    model,
    module,
    batches,
    dev,
    layers,
    qk_query_tokens,
    activation_frequency,
):
    """
    Aggregate functional metrics over batches, then append parameter-spectrum metrics.
    """
    H = int(model.config.num_attention_heads)
    accum = {
        li: {} for li in layers
    }
    batch_counts = {li: 0 for li in layers}

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)

            with LayerInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = call_model(
                        model,
                        batch,
                        pass_easiness=hasattr(
                            model.model.blocks[0], "mlt_vw_rtr"
                        ),
                    )
                dev.mark_step()

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn

                with dev.autocast():
                    q, k, v, context = reconstruct_qkv_context(
                        module, attn, hidden, attn_mask
                    )

                qk = qk_functional_metrics(
                    q, k, attn_mask, qk_query_tokens
                )
                out_blocks = output_weight_blocks(attn)
                ov = ov_magnitude_metrics(
                    v, context, out_blocks
                )

                all_metrics = {**qk, **ov}

                for name, tensor in all_metrics.items():
                    arr = np64(tensor)
                    if name not in accum[li]:
                        accum[li][name] = np.zeros(H, dtype=np.float64)
                    accum[li][name] += arr

                batch_counts[li] += 1

                del q, k, v, context

    rows = []

    for li in layers:
        attn = model.model.blocks[li].attn
        Wq, Wk, Wv = qkv_weight_blocks(attn)
        Wo = output_weight_blocks(attn)

        averaged = {
            k: v / max(1, batch_counts[li])
            for k, v in accum[li].items()
        }

        sqk = np64(
            (
                attn.sqk
                * (
                    attn.ngpt_sqk_init_value
                    / attn.ngpt_sqk_init_scale
                )
            ).view(H, attn.d_head)
        )

        for h in range(H):
            # QK operator: W_Q^T W_K
            qk_s = low_rank_product_singular_values(
                Wq[h].T, Wk[h]
            )
            qk_spec = spectrum_metrics(qk_s)

            # OV operator: W_O^(h) W_V^(h)
            ov_s = low_rank_product_singular_values(
                Wo[h], Wv[h]
            )
            ov_spec = spectrum_metrics(ov_s)

            row = {
                "model": label,
                "layer": li,
                "head": h,
                "activation_frequency": float(
                    activation_frequency[li,h]
                ),
                "sqk_rms": float(
                    np.sqrt(np.mean(np.square(sqk[h])))
                ),
                "sqk_mean_abs": float(
                    np.mean(np.abs(sqk[h]))
                ),
            }

            for name, values in averaged.items():
                row[name] = float(values[h])

            for kname, value in qk_spec.items():
                row[f"qk_operator_{kname}"] = value

            for kname, value in ov_spec.items():
                row[f"ov_operator_{kname}"] = value

            rows.append(row)

    return rows


# ---------------------------------------------------------------------------
# Current gradient probe
# ---------------------------------------------------------------------------

def extract_head_gradients(model, label, mode):
    rows = []
    H = int(model.config.num_attention_heads)

    for li, block in enumerate(model.model.blocks):
        attn = block.attn
        d = int(attn.d_head)
        D = int(attn.hidden_size)

        qkv_grad = attn.qkv.weight.grad
        out_grad = attn.output.weight.grad

        if qkv_grad is None:
            qg = kg = vg = torch.zeros(
                H, d, D, device=attn.qkv.weight.device
            )
        else:
            qg0, kg0, vg0 = qkv_grad.split(
                attn.total_head_dim, dim=0
            )
            qg = qg0.view(H,d,D)
            kg = kg0.view(H,d,D)
            vg = vg0.view(H,d,D)

        if out_grad is None:
            og = torch.zeros(
                H, D, d, device=attn.output.weight.device
            )
        else:
            og = out_grad.view(D,H,d).permute(1,0,2)

        qn = np64(qg.square().sum(dim=(1,2)).sqrt())
        kn = np64(kg.square().sum(dim=(1,2)).sqrt())
        vn = np64(vg.square().sum(dim=(1,2)).sqrt())
        on = np64(og.square().sum(dim=(1,2)).sqrt())

        for h in range(H):
            rows.append(
                {
                    "model": label,
                    "mode": mode,
                    "layer": li,
                    "head": h,
                    "grad_q_frobenius": float(qn[h]),
                    "grad_k_frobenius": float(kn[h]),
                    "grad_v_frobenius": float(vn[h]),
                    "grad_o_frobenius": float(on[h]),
                    "grad_qkvo_sum": float(
                        qn[h]+kn[h]+vn[h]+on[h]
                    ),
                }
            )

    return rows


def current_gradient_probe(
    model,
    label,
    cpu_batch,
    dev,
    is_7e,
):
    results = []

    def run(mode, override_context=None):
        model.zero_grad(set_to_none=True)
        model.eval()
        batch = move_batch(cpu_batch, dev)

        cm = override_context if override_context is not None else contextlib.nullcontext()
        with cm:
            with dev.autocast():
                logits = call_model(
                    model,
                    batch,
                    pass_easiness=is_7e,
                )
                loss = per_example_ce(
                    logits, batch["labels"]
                ).mean()

            loss.backward()
            dev.mark_step()

        rows = extract_head_gradients(model, label, mode)
        for r in rows:
            r["probe_ce"] = float(
                loss.detach().to(torch.float32).cpu()
            )
        return rows

    if is_7e:
        results.extend(run("routed"))
        results.extend(
            run(
                "forced_dense_all32",
                LearnedRouterOverride(model, "dense"),
            )
        )
    else:
        results.extend(run("dense"))

    model.zero_grad(set_to_none=True)
    return results


# ---------------------------------------------------------------------------
# Layer comparison / plots
# ---------------------------------------------------------------------------

def aggregate_layer(rows):
    out = []
    numeric_keys = [
        k for k in rows[0].keys()
        if k not in {"model","layer","head"}
        and isinstance(rows[0][k], (int,float,np.integer,np.floating))
    ]

    for model in sorted(set(r["model"] for r in rows)):
        for li in sorted(set(int(r["layer"]) for r in rows if r["model"] == model)):
            rr = [
                r for r in rows
                if r["model"] == model and int(r["layer"]) == li
            ]
            row = {"model": model, "layer": li}
            for k in numeric_keys:
                vals = [
                    float(r[k]) for r in rr
                    if np.isfinite(float(r[k]))
                ]
                if vals:
                    row[f"{k}_mean"] = float(np.mean(vals))
                    row[f"{k}_std_across_heads"] = float(np.std(vals))
            out.append(row)
    return out


def paired_layer_comparison(layer_rows):
    models = sorted(set(r["model"] for r in layer_rows))
    if len(models) != 2:
        return []

    a, b = models
    amap = {(r["layer"]): r for r in layer_rows if r["model"] == a}
    bmap = {(r["layer"]): r for r in layer_rows if r["model"] == b}

    rows = []
    for li in sorted(set(amap) & set(bmap)):
        row = {
            "layer": li,
            "model_a": a,
            "model_b": b,
        }
        common = set(amap[li]) & set(bmap[li])
        for k in common:
            if k in {"model","layer"}:
                continue
            va = amap[li][k]
            vb = bmap[li][k]
            if isinstance(va,(int,float)) and isinstance(vb,(int,float)):
                row[f"{k}_a"] = va
                row[f"{k}_b"] = vb
                row[f"{k}_a_minus_b"] = va - vb
        rows.append(row)
    return rows


def make_plots(layer_rows, out):
    metrics = [
        ("normalized_attention_entropy_mean", "Normalized attention entropy"),
        ("max_attention_probability_mean", "Max attention probability"),
        ("qk_logit_std_mean", "QK logit std"),
        ("value_rms_mean", "Value RMS"),
        ("context_rms_mean", "Context RMS"),
        ("residual_rms_mean", "Residual-write RMS"),
        ("output_block_frobenius_mean", "W_O block Frobenius"),
        ("alignment_gain_mean", "Context/W_O alignment gain"),
        ("ov_operator_stable_rank_mean", "OV operator stable rank"),
    ]

    models = sorted(set(r["model"] for r in layer_rows))

    for metric, ylabel in metrics:
        fig, ax = plt.subplots(figsize=(8,4.5))
        used = False

        for model in models:
            rr = sorted(
                [r for r in layer_rows if r["model"] == model],
                key=lambda x: int(x["layer"]),
            )
            xs = [int(r["layer"]) for r in rr if metric in r]
            ys = [float(r[metric]) for r in rr if metric in r]
            if xs:
                ax.plot(xs, ys, marker="o", label=model)
                used = True

        if not used:
            plt.close(fig)
            continue

        ax.set_xlabel("Layer")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel + " by depth")
        ax.legend()
        fig.tight_layout()
        filename = metric.replace("_mean","") + "_by_depth.png"
        fig.savefig(out / filename, dpi=180)
        plt.close(fig)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--device", default="auto", choices=["auto","cuda","xla","cpu"])

    ap.add_argument("--model-7e", default="model_7e.py")
    ap.add_argument("--model-dense", default="model_dense32.py")

    ap.add_argument("--repo-7e", default=DEFAULT_7E_REPO)
    ap.add_argument("--repo-dense", default=DEFAULT_DENSE_REPO)
    ap.add_argument("--checkpoint-7e", default=CHECKPOINT_FILE)
    ap.add_argument("--checkpoint-dense", default=CHECKPOINT_FILE)
    ap.add_argument("--checkpoint-7e-path", default=None)
    ap.add_argument("--checkpoint-dense-path", default=None)

    ap.add_argument("--validation-path", default=None)
    ap.add_argument("--data-repo", default=DATA_REPO)
    ap.add_argument("--validation-file", default=VALIDATION_FILE)

    ap.add_argument("--num-examples", type=int, default=16)
    ap.add_argument("--batch-size", type=int, default=2)
    ap.add_argument("--seq-len", type=int, default=1024)
    ap.add_argument("--layers", default="0,1,2,3,4,5,6,7,8,9,10,11")
    ap.add_argument("--qk-query-tokens", type=int, default=64)

    ap.add_argument(
        "--gradient-probe",
        action="store_true",
        help="Run one-batch current Q/K/V/O gradient probe.",
    )

    ap.add_argument("--seed", type=int, default=1216)
    ap.add_argument("--output-dir", default="helm_7e_vs_dense32_attention_anatomy")
    ap.add_argument("--cache-dir", default=".helm_attention_anatomy_cache")
    args = ap.parse_args()

    seed_everything(args.seed)
    out = Path(args.output_dir)
    cache = Path(args.cache_dir)
    out.mkdir(parents=True, exist_ok=True)
    cache.mkdir(parents=True, exist_ok=True)

    dev = resolve_device(args.device)
    token = get_hf_token()

    mod7 = import_model_file(
        Path(args.model_7e), "helm7e_anatomy_arch"
    )
    modd = import_model_file(
        Path(args.model_dense), "dense32_anatomy_arch"
    )

    # Keep repositories in completely separate cache directories.
    # Both repositories use filenames like checkpoint-006500.pt, so sharing
    # one local_dir would cause one checkpoint to overwrite/reuse the other.
    cache_7e = cache / "7e"
    cache_dense = cache / "dense32"
    
    ck7 = resolve_checkpoint(
        args.checkpoint_7e_path,
        args.repo_7e,
        args.checkpoint_7e,
        cache_7e,
        token,
    )
    
    state7 = maybe_training_state(
        args.repo_7e,
        cache_7e,
        token,
    )
    breakpoints = read_breakpoints(state7)
    
    # Load 7e from its own checkpoint/cache.
    model7, cfg7 = load_model(
        mod7,
        ck7,
        dev,
        breakpoints,
    )
    
    ckd = resolve_checkpoint(
        args.checkpoint_dense_path,
        args.repo_dense,
        args.checkpoint_dense,
        cache_dense,
        token,
    )
    
    # Load Dense-32 independently.
    modeld, cfgd = load_model(
        modd,
        ckd,
        dev,
        None,
    )

    validation = resolve_validation(
        args.validation_path,
        args.data_repo,
        args.validation_file,
        cache,
        token,
    )

    # Same token IDs / MLM corruption for both models.
    examples = prepare_examples(
        validation,
        cfg7,
        args.num_examples,
        args.seq_len,
        args.seed,
    )
    examples, batches = make_batches(
        examples, args.batch_size
    )

    layers = parse_layers(args.layers)

    print("\n=== Activation frequencies ===")
    af7 = capture_activation_frequency(
        model7, batches, dev
    )
    afd = np.ones(
        (
            len(modeld.model.blocks),
            int(modeld.config.num_attention_heads),
        ),
        dtype=np.float64,
    )

    print("\n=== HELM_7e anatomy ===")
    rows7 = analyze_model(
        "HELM_7e",
        model7,
        mod7,
        batches,
        dev,
        layers,
        args.qk_query_tokens,
        af7,
    )

    print("\n=== Dense-32 anatomy ===")
    rowsd = analyze_model(
        "Dense-32",
        modeld,
        modd,
        batches,
        dev,
        layers,
        args.qk_query_tokens,
        afd,
    )

    rows = rows7 + rowsd
    write_csv(out / "head_attention_anatomy.csv", rows)
    write_csv(out / "head_attention_anatomy_7e.csv", rows7)
    write_csv(out / "head_attention_anatomy_dense32.csv", rowsd)

    layer_rows = aggregate_layer(rows)
    write_csv(out / "layer_attention_anatomy.csv", layer_rows)

    paired = paired_layer_comparison(layer_rows)
    write_csv(out / "layer_7e_minus_dense32.csv", paired)

    grad_rows = []
    if args.gradient_probe:
        print("\n=== Current gradient probe ===")
        grad_rows.extend(
            current_gradient_probe(
                model7,
                "HELM_7e",
                batches[0],
                dev,
                is_7e=True,
            )
        )
        grad_rows.extend(
            current_gradient_probe(
                modeld,
                "Dense-32",
                batches[0],
                dev,
                is_7e=False,
            )
        )
        write_csv(out / "current_gradient_probe.csv", grad_rows)

    make_plots(layer_rows, out)

    # Correlations that directly address magnitude origin and 7e routing.
    corr_rows = []
    for model_name, rr in (
        ("HELM_7e", rows7),
        ("Dense-32", rowsd),
    ):
        for li in layers:
            x = [r for r in rr if int(r["layer"]) == li]
            if not x:
                continue

            def corr(a,b):
                return spearman_np(
                    [r[a] for r in x],
                    [r[b] for r in x],
                )

            corr_rows.append(
                {
                    "model": model_name,
                    "layer": li,
                    "rho_residualRMS_contextRMS": corr(
                        "residual_rms","context_rms"
                    ),
                    "rho_residualRMS_WO": corr(
                        "residual_rms","output_block_frobenius"
                    ),
                    "rho_residualRMS_alignment": corr(
                        "residual_rms","alignment_gain"
                    ),
                    "rho_activation_residualRMS": corr(
                        "activation_frequency","residual_rms"
                    ),
                    "rho_activation_contextRMS": corr(
                        "activation_frequency","context_rms"
                    ),
                    "rho_activation_WO": corr(
                        "activation_frequency","output_block_frobenius"
                    ),
                    "rho_activation_QKentropy": corr(
                        "activation_frequency",
                        "normalized_attention_entropy",
                    ),
                }
            )
    write_csv(out / "head_metric_correlations.csv", corr_rows)

    # Human-readable summary with layer means for key mechanisms.
    lines = [
        "# HELM_7e vs Dense-32 attention anatomy",
        "",
        "## Interpretation",
        "",
        "- Lower QK entropy = more concentrated reading, but is not automatically better.",
        "- `attention_mixing_gain_power = context_power / value_power` measures how attention mixing changes V energy.",
        "- `output_block_frobenius` measures head-specific W_O block strength.",
        "- `alignment_gain = 1` is the isotropic-context reference. >1 means C_h covariance aligns with strong W_O directions; <1 means it avoids them.",
        "- `OV operator stable rank` describes the intrinsic low-rank write transform, not output magnitude.",
        "- Current gradient probe (if enabled) is local checkpoint information, NOT training history.",
        "",
        "## Key layer averages",
        "",
        "| model | L | QK norm entropy | max attn p | V RMS | C RMS | Y RMS | W_O norm | align gain | OV stable rank |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]

    for r in layer_rows:
        lines.append(
            f"| {r['model']} | {r['layer']} | "
            f"{r.get('normalized_attention_entropy_mean',float('nan')):.4f} | "
            f"{r.get('max_attention_probability_mean',float('nan')):.4f} | "
            f"{r.get('value_rms_mean',float('nan')):.5f} | "
            f"{r.get('context_rms_mean',float('nan')):.5f} | "
            f"{r.get('residual_rms_mean',float('nan')):.5f} | "
            f"{r.get('output_block_frobenius_mean',float('nan')):.4f} | "
            f"{r.get('alignment_gain_mean',float('nan')):.4f} | "
            f"{r.get('ov_operator_stable_rank_mean',float('nan')):.2f} |"
        )

    lines += [
        "",
        "## What to look for",
        "",
        "### If HELM_7e has normal QK metrics but low residual RMS",
        "The deficit is primarily on the WRITE side, not attention selection.",
        "",
        "### If C RMS is already much lower in HELM_7e",
        "Magnitude is being lost inside V/attention mixing before W_O.",
        "",
        "### If C RMS is similar but W_O norms are much smaller/more unequal",
        "The write projection is the main magnitude bottleneck; head-normalized W_O / explicit head gates become a targeted ablation.",
        "",
        "### If C RMS and W_O norms are similar but alignment_gain is lower",
        "The issue is orientation: head contexts are poorly matched to their W_O subspaces.",
        "",
        "### If QK entropy/logit contrast differs strongly",
        "The router is changing what heads learn to READ, so an MLP/nonlinear head-combination fix may be downstream of the true problem.",
    ]

    (out / "summary.md").write_text("\n".join(lines))

    z = zip_results(
        out,
        "helm_7e_vs_dense32_attention_anatomy_results",
    )
    print(f"\nDone: {z}")


if __name__ == "__main__":
    main()

Overwriting analyze_7e_vs_dense32_attention_anatomy.py


In [6]:
%%writefile helm_analysis_core.py
"""
Shared analysis utilities for HELM routed-head experiments.

IMPORTANT NUMPY CONVERSION RULE
-------------------------------
Every Gram matrix is converted in exactly this order:

    tensor.detach().to(torch.float32).cpu().numpy().astype(np.float64)

Do not reorder that conversion.

The three companion scripts intentionally use the same:
- deterministic validation examples
- MLM masking
- CE implementation
- context/residual head geometry
- raw/directional effective-rank calculations

so their outputs are directly comparable.
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import importlib.util
import json
import math
import os
import random
import shutil
import sys
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError("pyarrow is required") from exc

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError("huggingface_hub is required") from exc


DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"
CHECKPOINT_FILE = "checkpoint-006500.pt"
TRAINING_STATE_FILE = "training_state.json"


# ---------------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------------

def get_hf_token():
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def rankdata_np(x):
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=np.float64)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        ranks[order[i:j]] = 0.5 * (i + j - 1)
        i = j
    return ranks


def spearman_np(x, y):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(rankdata_np(x), rankdata_np(y))[0, 1])


def safe_mean(x):
    a = np.asarray(list(x), dtype=np.float64)
    a = a[np.isfinite(a)]
    return float(a.mean()) if len(a) else float("nan")


def safe_std(x):
    a = np.asarray(list(x), dtype=np.float64)
    a = a[np.isfinite(a)]
    return float(a.std(ddof=1)) if len(a) > 1 else float("nan")


def safe_sem(x):
    a = np.asarray(list(x), dtype=np.float64)
    a = a[np.isfinite(a)]
    return float(a.std(ddof=1) / math.sqrt(len(a))) if len(a) > 1 else float("nan")


def gini_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, 0.0, None)
    if x.sum() <= 0 or len(x) == 0:
        return 0.0
    sx = np.sort(x)
    n = len(sx)
    return float(
        (2.0 * np.sum((np.arange(1, n + 1)) * sx) / (n * sx.sum()))
        - (n + 1) / n
    )


def effective_rank_from_gram(gram):
    gram = np.asarray(gram, dtype=np.float64)
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)
    total = vals.sum()
    if total <= 1e-18:
        return 0.0, 0.0, vals
    p = vals / total
    pp = p[p > 1e-15]
    erank = float(np.exp(-(pp * np.log(pp)).sum()))
    prank = float((total * total) / (np.square(vals).sum() + 1e-18))
    return erank, prank, vals


def directional_gram(raw_gram):
    raw_gram = np.asarray(raw_gram, dtype=np.float64)
    diag = np.clip(np.diag(raw_gram), 1e-18, None)
    denom = np.sqrt(np.outer(diag, diag))
    corr = raw_gram / denom
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def mean_abs_offdiag(mat):
    mat = np.asarray(mat, dtype=np.float64)
    if mat.shape[0] <= 1:
        return 0.0
    mask = ~np.eye(mat.shape[0], dtype=bool)
    return float(np.abs(mat[mask]).mean())


def parse_layers(text):
    return [int(x.strip()) for x in text.split(",") if x.strip()]


def write_csv(path: Path, rows: List[dict]):
    if not rows:
        path.write_text("")
        return
    with path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def strip_state_prefixes(state):
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def import_model_file(path: Path, module_name="analysis_arch"):
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import {path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod


# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------

@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast("cuda", dtype=torch.bfloat16)
        if self.kind == "xla":
            return torch.autocast("xla", dtype=torch.bfloat16)
        return contextlib.nullcontext()


def resolve_device(requested):
    requested = requested.lower()
    if requested == "auto":
        if torch.cuda.is_available():
            requested = "cuda"
        else:
            try:
                import torch_xla.core.xla_model as xm
                return DeviceContext(xm.xla_device(), "xla", xm)
            except Exception:
                requested = "cpu"

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable")
        return DeviceContext(torch.device("cuda"), "cuda")

    if requested == "xla":
        import torch_xla.core.xla_model as xm
        return DeviceContext(xm.xla_device(), "xla", xm)

    if requested == "cpu":
        return DeviceContext(torch.device("cpu"), "cpu")

    raise ValueError(requested)


# ---------------------------------------------------------------------------
# Assets/model/data
# ---------------------------------------------------------------------------

def download_file(repo, filename, repo_type, cache_dir, token, label):
    print(f"Downloading {label}: {repo}/{filename}")
    return Path(
        hf_hub_download(
            repo_id=repo,
            filename=filename,
            repo_type=repo_type,
            token=token,
            local_dir=str(cache_dir / label),
        )
    )


def resolve_checkpoint(local_path, repo, filename, cache_dir, token):
    if local_path:
        p = Path(local_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    return download_file(repo, filename, "model", cache_dir, token, "checkpoint")


def resolve_validation(local_path, repo, filename, cache_dir, token):
    if local_path:
        p = Path(local_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    return download_file(repo, filename, "dataset", cache_dir, token, "validation")


def maybe_training_state(repo, cache_dir, token):
    try:
        return download_file(
            repo, TRAINING_STATE_FILE, "model", cache_dir, token, "training_state"
        )
    except Exception as exc:
        print(f"WARNING: training_state.json unavailable: {exc}")
        return None


def read_breakpoints(path):
    if path is None:
        return None
    try:
        state = json.loads(Path(path).read_text())
        ed = state.get("easiness_dict")
        if isinstance(ed, dict) and ed.get("breakpoints"):
            return [float(x) for x in ed["breakpoints"]]
    except Exception:
        pass
    return None


def instantiate_config(module, breakpoints=None):
    kwargs = {}
    if breakpoints is not None:
        kwargs["easiness_cdf_breakpoints"] = breakpoints
    try:
        return module.HELMConfig(**kwargs)
    except TypeError:
        return module.HELMConfig()


def load_model(module, checkpoint, dev, breakpoints=None):
    cfg = instantiate_config(module, breakpoints)
    model = module.HELMForMaskedLM(cfg)
    payload = torch.load(str(checkpoint), map_location="cpu")
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    state = strip_state_prefixes(state)

    incompat = model.load_state_dict(state, strict=False)
    missing = list(incompat.missing_keys)
    unexpected = list(incompat.unexpected_keys)
    if missing or unexpected:
        print(f"State load: {len(missing)} missing, {len(unexpected)} unexpected")
        if missing:
            print(" missing:", missing[:8])
        if unexpected:
            print(" unexpected:", unexpected[:8])
        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError("Large checkpoint/architecture mismatch")

    model.to(dev.device)
    model.eval()
    if hasattr(model, "enable_efficient_inference"):
        try:
            model.enable_efficient_inference("dense", compile=False)
        except Exception:
            pass
    return model, cfg


def deterministic_span_mask(
    ids,
    config,
    seed,
    probability=0.30,
    span_length=3,
):
    ids = ids.clone().long()
    labels = torch.full_like(ids, -100)
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))

    special = {
        int(config.bos_token_id),
        int(config.eos_token_id),
        int(config.pad_token_id),
        int(config.mask_token_id),
        int(config.unk_token_id),
    }
    candidates = [i for i, t in enumerate(ids.tolist()) if int(t) not in special]
    if not candidates:
        return ids, labels

    target = max(1, int(round(probability * len(candidates))))
    cset = set(candidates)
    perm = torch.randperm(len(candidates), generator=g).tolist()
    chosen = set()

    for pi in perm:
        if len(chosen) >= target:
            break
        st = candidates[pi]
        for p in range(st, min(st + span_length, ids.numel())):
            if p in cset:
                chosen.add(p)
                if len(chosen) >= target:
                    break

    pos = torch.tensor(sorted(chosen), dtype=torch.long)
    labels[pos] = ids[pos]
    r = torch.rand(len(pos), generator=g)

    mask_sel = r < 0.80
    rand_sel = (r >= 0.80) & (r < 0.90)
    ids[pos[mask_sel]] = int(config.mask_token_id)

    if rand_sel.any():
        ids[pos[rand_sel]] = torch.randint(
            0,
            int(config.vocab_size),
            (int(rand_sel.sum()),),
            generator=g,
        )
    return ids, labels


def prepare_examples(validation_path, config, num_examples, seq_len, seed):
    table = pq.read_table(
        str(validation_path), columns=["input_ids", "easiness_score"]
    )
    n = min(int(num_examples), table.num_rows)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(table.num_rows)[:n]

    input_col = table.column("input_ids")
    easy_col = table.column("easiness_score")
    examples = []

    for i, row_idx in enumerate(indices.tolist()):
        ids = torch.tensor(input_col[row_idx].as_py(), dtype=torch.long)[:seq_len]
        if ids.numel() < seq_len:
            ids = torch.cat(
                [
                    ids,
                    torch.full(
                        (seq_len - ids.numel(),),
                        int(config.pad_token_id),
                        dtype=torch.long,
                    ),
                ]
            )

        masked, labels = deterministic_span_mask(
            ids, config, seed=seed + 100003 * i
        )
        examples.append(
            {
                "input_ids": masked,
                "labels": labels,
                "attention_mask": (masked != int(config.pad_token_id)).long(),
                "easiness_score": torch.tensor(
                    float(easy_col[row_idx].as_py()), dtype=torch.float32
                ),
                "example_id": torch.tensor(i, dtype=torch.long),
            }
        )
    return examples


def make_batches(examples, batch_size):
    usable = (len(examples) // batch_size) * batch_size
    examples = examples[:usable]
    if not examples:
        raise RuntimeError("Not enough complete examples for one batch")

    batches = []
    for s in range(0, usable, batch_size):
        chunk = examples[s : s + batch_size]
        batches.append(
            {
                k: torch.stack([x[k] for x in chunk], dim=0)
                for k in chunk[0]
            }
        )
    return examples, batches


def make_subbatches(examples, indices, batch_size):
    idx = list(indices)
    usable = (len(idx) // batch_size) * batch_size
    idx = idx[:usable]
    out = []
    for s in range(0, usable, batch_size):
        chunk = [examples[i] for i in idx[s : s + batch_size]]
        out.append(
            {
                k: torch.stack([x[k] for x in chunk], dim=0)
                for k in chunk[0]
            }
        )
    return out


def move_batch(batch, dev):
    return {k: v.to(dev.device) for k, v in batch.items()}


# ---------------------------------------------------------------------------
# Forward / CE
# ---------------------------------------------------------------------------

def call_model(model, batch, pass_easiness=True, current_step=6500):
    kwargs = {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
    }
    if pass_easiness:
        kwargs["easiness_score"] = batch.get("easiness_score")
    else:
        kwargs["easiness_score"] = None
    kwargs["current_step"] = current_step

    try:
        out = model(**kwargs)
    except TypeError:
        kwargs.pop("current_step", None)
        try:
            out = model(**kwargs)
        except TypeError:
            kwargs.pop("easiness_score", None)
            out = model(**kwargs)

    return out[0] if isinstance(out, (tuple, list)) else out


def per_example_ce(logits, labels, chunk_tokens=128):
    B, S, V = logits.shape
    sums = torch.zeros(B, device=logits.device, dtype=torch.float32)
    counts = torch.zeros(B, device=logits.device, dtype=torch.float32)

    for s in range(0, S, chunk_tokens):
        e = min(s + chunk_tokens, S)
        lgt = logits[:, s:e, :].to(torch.float32)
        lab = labels[:, s:e]
        losses = F.cross_entropy(
            lgt.reshape(-1, V),
            lab.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(B, e - s)
        valid = (lab != -100).float()
        sums += (losses * valid).sum(dim=1)
        counts += valid.sum(dim=1)

    return sums / counts.clamp_min(1.0)


def evaluate_ce(
    model,
    batches,
    dev,
    pass_easiness=True,
    return_per_example=False,
):
    all_vals = {}
    total_sum = 0.0
    total_n = 0

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                logits = call_model(
                    model, batch, pass_easiness=pass_easiness
                )
                vals = per_example_ce(logits, batch["labels"])
            dev.mark_step()

            vals_cpu = vals.detach().to(torch.float32).cpu().tolist()
            ids = cpu_batch["example_id"].tolist()
            for eid, val in zip(ids, vals_cpu):
                all_vals[int(eid)] = float(val)
                total_sum += float(val)
                total_n += 1

    mean = total_sum / max(1, total_n)
    return (mean, all_vals) if return_per_example else mean


# ---------------------------------------------------------------------------
# Mask adapters
# ---------------------------------------------------------------------------

def full_mask_from_model(model, variant):
    """Return [B,L,H] hard-forward masks from the most recent forward."""
    layers = []

    if variant in {"learned", "topk"}:
        P = int(model.config.num_permanent_heads)
        for block in model.model.blocks:
            elastic = (
                block.mlt_vw_rtr.save_hard_mask.detach()
                .to(torch.float32)
                .cpu()
            )
            perm = torch.ones(
                elastic.size(0), P, dtype=torch.float32
            )
            layers.append(torch.cat([perm, elastic], dim=-1))

    elif variant == "random":
        for block in model.model.blocks:
            m = (
                block.attn.save_random_mask.detach()
                .to(torch.float32)
                .cpu()
            )
            layers.append(m)

    else:
        raise ValueError(variant)

    return torch.stack(layers, dim=1)


def collect_masks_and_ce(
    model,
    batches,
    dev,
    variant,
    pass_easiness,
):
    masks = {}
    ce_by_id = {}

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                logits = call_model(
                    model, batch, pass_easiness=pass_easiness
                )
                vals = per_example_ce(logits, batch["labels"])
            dev.mark_step()

            fm = full_mask_from_model(model, variant)
            ids = cpu_batch["example_id"].tolist()
            vals = vals.detach().to(torch.float32).cpu().tolist()

            for bi, eid in enumerate(ids):
                masks[int(eid)] = fm[bi].numpy().astype(np.float32)
                ce_by_id[int(eid)] = float(vals[bi])

    return masks, ce_by_id


class LearnedRouterOverride:
    """
    Forward-hook replacement for HELM_7c/7e/constant-K router outputs.

    mode:
      dense
      permanent
      random_same_count
      dense_minus (requires target layer/head)
      forced (requires dict layer->[B,H] full masks)
    """

    def __init__(
        self,
        model,
        mode,
        target_layer=None,
        target_head=None,
        forced_masks=None,
    ):
        self.model = model
        self.mode = mode
        self.target_layer = target_layer
        self.target_head = target_head
        self.forced_masks = forced_masks or {}
        self.handles = []

    def _hook(self, li):
        def hook(module, inputs, output):
            B, H, _, _ = output.shape
            P = int(self.model.config.num_permanent_heads)

            if self.mode == "dense":
                return torch.ones_like(output)

            if self.mode == "permanent":
                result = torch.zeros_like(output)
                result[:, :P, :, :] = 1
                return result

            if self.mode == "dense_minus":
                result = torch.ones_like(output)
                if li == self.target_layer:
                    result[:, int(self.target_head), :, :] = 0
                return result

            if self.mode == "random_same_count":
                # output forward values are hard 0/1 even though an STE exists.
                k_elastic = (
                    output[:, P:, 0, 0].detach().float().sum(dim=-1).long()
                )
                E = H - P
                scores = torch.rand(
                    B, E, device=output.device, dtype=torch.float32
                )
                # Random unique ranks 0..E-1 per example.
                order = torch.argsort(scores, dim=-1, descending=True)
                ranks = torch.argsort(order, dim=-1)
                elastic = (
                    ranks < k_elastic.view(B, 1)
                ).to(output.dtype)
                perm = torch.ones(
                    B, P, device=output.device, dtype=output.dtype
                )
                full = torch.cat([perm, elastic], dim=-1)
                return full.view(B, H, 1, 1)

            if self.mode == "forced":
                m = self.forced_masks[li].to(
                    device=output.device, dtype=output.dtype
                )
                return m.view(B, H, 1, 1)

            raise ValueError(self.mode)

        return hook

    def __enter__(self):
        for li, block in enumerate(self.model.model.blocks):
            self.handles.append(
                block.mlt_vw_rtr.register_forward_hook(self._hook(li))
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


class RandomAttentionMaskOverride:
    """
    Override HELM_32_random._random_full_mask for controlled CE tests.

    mode:
      dense
      permanent
      dense_minus
      forced
    """

    def __init__(
        self,
        model,
        mode,
        target_layer=None,
        target_head=None,
        forced_masks=None,
    ):
        self.model = model
        self.mode = mode
        self.target_layer = target_layer
        self.target_head = target_head
        self.forced_masks = forced_masks or {}
        self.originals = []
        self.old_modes = []

    def __enter__(self):
        P = int(self.model.config.num_permanent_heads)
        H = int(self.model.config.num_attention_heads)

        for li, block in enumerate(self.model.model.blocks):
            attn = block.attn
            self.old_modes.append(attn.routing_mode)
            attn.set_routing_mode("random")
            original = attn._random_full_mask
            self.originals.append(original)

            def make_override(layer_idx, attn_obj):
                def fn(_self, batch_size, device, dtype):
                    if self.mode == "dense":
                        return torch.ones(
                            batch_size, H, device=device, dtype=dtype
                        )

                    if self.mode == "permanent":
                        result = torch.zeros(
                            batch_size, H, device=device, dtype=dtype
                        )
                        result[:, :P] = 1
                        return result

                    if self.mode == "dense_minus":
                        result = torch.ones(
                            batch_size, H, device=device, dtype=dtype
                        )
                        if layer_idx == self.target_layer:
                            result[:, int(self.target_head)] = 0
                        return result

                    if self.mode == "forced":
                        return self.forced_masks[layer_idx].to(
                            device=device, dtype=dtype
                        )

                    raise ValueError(self.mode)
                return types.MethodType(fn, attn_obj)

            attn._random_full_mask = make_override(li, attn)

        return self

    def __exit__(self, exc_type, exc, tb):
        for block, original, old_mode in zip(
            self.model.model.blocks, self.originals, self.old_modes
        ):
            block.attn._random_full_mask = original
            block.attn.set_routing_mode(old_mode)


def forced_masks_for_ids(masks_by_id, ids, dev, target_layer, target_head, value):
    L, H = next(iter(masks_by_id.values())).shape
    out = {}
    for li in range(L):
        arr = np.stack([masks_by_id[int(i)][li] for i in ids], axis=0)
        if li == target_layer:
            arr[:, target_head] = float(value)
        out[li] = torch.tensor(arr, dtype=torch.float32, device=dev.device)
    return out


# ---------------------------------------------------------------------------
# Attention geometry
# ---------------------------------------------------------------------------

class AttentionInputCapture:
    def __init__(self, model, layers):
        self.model = model
        self.layers = list(layers)
        self.data = {}
        self.handles = []

    def _hook(self, li):
        def hook(module, inputs):
            # All current models have hidden_states and attention_mask first.
            self.data[li] = (inputs[0].detach(), inputs[1].detach())
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(
                    self._hook(li)
                )
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def unmasked_attention_context(module, attn, hidden_states, attention_mask):
    qkv_proj = module.cast_linear(hidden_states, attn.qkv)
    B, S, _ = hidden_states.shape

    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)
    q = q.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    k = k.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    v = v.view(B, S, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)

    q = module.justnorm(q)
    k = module.justnorm(k)
    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = attn.sqk * (
        attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale
    )
    sqk = sqk.view(
        1, attn.num_attention_heads, 1, attn.d_head
    ).to(q.dtype)

    q = sqk * q
    k = sqk * k

    context = F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )

    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (
            context * vn
        ).sum(dim=-1, keepdim=True) * vn

    return context


def add_gram(acc, key, value):
    if key not in acc:
        acc[key] = np.zeros_like(value, dtype=np.float64)
    acc[key] += value


def geometry_analysis(
    model,
    module,
    batches,
    dev,
    variant,
    pass_easiness,
    layers,
    max_batches,
    sample_tokens,
):
    """
    Produce potential and executed context/residual Gram matrices.

    IMPORTANT:
      Conversion is exactly:
      detach -> float32 -> cpu -> numpy -> float64
    """
    grams = {}
    pr_ratio_examples = {li: [] for li in layers}

    with torch.no_grad():
        for cpu_batch in batches[:max_batches]:
            batch = move_batch(cpu_batch, dev)

            with AttentionInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = call_model(
                        model, batch, pass_easiness=pass_easiness
                    )
                dev.mark_step()

            full_masks = full_mask_from_model(model, variant).to(dev.device)

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn

                with dev.autocast():
                    context = unmasked_attention_context(
                        module, attn, hidden, attn_mask
                    )  # [B,H,S,d]

                    S = context.size(2)
                    T = min(int(sample_tokens), S)
                    positions = torch.linspace(
                        0, S - 1, steps=T, device=context.device
                    ).long()
                    c = context.index_select(2, positions)

                    mask = full_masks[:, li, :].to(
                        device=c.device, dtype=c.dtype
                    ).view(c.size(0), c.size(1), 1, 1)

                    c_exec = c * mask

                    W = attn.output.weight.to(c.dtype).view(
                        attn.hidden_size,
                        attn.num_attention_heads,
                        attn.d_head,
                    )

                    y = torch.einsum("bhtd,ohd->bhto", c, W)
                    y_exec = y * mask

                    # Aggregated Gram matrices.
                    for state, c_use, y_use in (
                        ("potential", c, y),
                        ("executed", c_exec, y_exec),
                    ):
                        cflat = (
                            c_use.permute(1, 0, 2, 3)
                            .contiguous()
                            .view(c_use.size(1), -1)
                        )
                        yflat = (
                            y_use.permute(1, 0, 2, 3)
                            .contiguous()
                            .view(y_use.size(1), -1)
                        )

                        cgram = cflat @ cflat.T
                        ygram = yflat @ yflat.T

                        # REQUIRED CONVERSION ORDER.
                        cgram_np = (
                            cgram.detach()
                            .to(torch.float32)
                            .cpu()
                            .numpy()
                            .astype(np.float64)
                        )
                        ygram_np = (
                            ygram.detach()
                            .to(torch.float32)
                            .cpu()
                            .numpy()
                            .astype(np.float64)
                        )

                        add_gram(grams, (li, state, "context"), cgram_np)
                        add_gram(grams, (li, state, "residual"), ygram_np)

                    # Exact per-example context participation-rank ratio r_PR / K.
                    flat = c_exec.to(torch.float32).reshape(
                        c_exec.size(0), c_exec.size(1), -1
                    )
                    flat = flat / math.sqrt(float(max(1, flat.size(-1))))
                    g = torch.bmm(flat, flat.transpose(1, 2))
                    tr = torch.diagonal(g, dim1=-2, dim2=-1).sum(-1)
                    tr2 = g.square().sum(dim=(-2, -1))
                    rpr = tr.square() / (tr2 + 1e-12)
                    K = full_masks[:, li, :].sum(-1).clamp_min(1).to(rpr.device)
                    ratio = rpr / K
                    pr_ratio_examples[li].extend(
                        ratio.detach().to(torch.float32).cpu().tolist()
                    )

    rows = []

    for key, gram in sorted(grams.items()):
        li, state, space = key
        raw_er, raw_pr, _ = effective_rank_from_gram(gram)
        dgram = directional_gram(gram)
        dir_er, dir_pr, _ = effective_rank_from_gram(dgram)

        energy = np.clip(np.diag(gram), 0.0, None)
        ef = energy / max(1e-18, energy.sum())
        top_sorted = np.sort(ef)[::-1]

        rows.append(
            {
                "layer": li,
                "state": state,
                "space": space,
                "heads": gram.shape[0],
                "raw_entropy_rank": raw_er,
                "raw_entropy_ratio": raw_er / gram.shape[0],
                "raw_participation_rank": raw_pr,
                "raw_participation_ratio": raw_pr / gram.shape[0],
                "directional_entropy_rank": dir_er,
                "directional_entropy_ratio": dir_er / gram.shape[0],
                "directional_participation_rank": dir_pr,
                "directional_participation_ratio": dir_pr / gram.shape[0],
                "mean_abs_cos": mean_abs_offdiag(dgram),
                "energy_gini": gini_np(energy),
                "top1_energy_fraction": float(top_sorted[:1].sum()),
                "top4_energy_fraction": float(top_sorted[:4].sum()),
            }
        )

    ratio_rows = []
    for li, vals in pr_ratio_examples.items():
        ratio_rows.append(
            {
                "layer": li,
                "mean_context_pr_ratio_per_example": safe_mean(vals),
                "std_context_pr_ratio_per_example": safe_std(vals),
                "min_context_pr_ratio_per_example": float(np.min(vals)) if vals else float("nan"),
                "max_context_pr_ratio_per_example": float(np.max(vals)) if vals else float("nan"),
            }
        )

    return rows, ratio_rows


# ---------------------------------------------------------------------------
# Router/mask statistics
# ---------------------------------------------------------------------------

def mask_statistics(masks_by_id, model):
    ids = sorted(masks_by_id)
    arr = np.stack([masks_by_id[i] for i in ids], axis=0)  # [N,L,H]
    N, L, H = arr.shape
    P = int(getattr(model.config, "num_permanent_heads", 0))

    layer_rows = []
    head_rows = []

    for li in range(L):
        counts = arr[:, li, :].sum(axis=-1)
        unique = np.unique(arr[:, li, :], axis=0).shape[0]

        layer_rows.append(
            {
                "layer": li,
                "mean_total_heads": float(counts.mean()),
                "std_total_heads": float(counts.std()),
                "min_total_heads": float(counts.min()),
                "max_total_heads": float(counts.max()),
                "unique_masks": int(unique),
                "unique_mask_fraction": float(unique / N),
            }
        )

        for h in range(H):
            head_rows.append(
                {
                    "layer": li,
                    "head": h,
                    "is_permanent_slot": int(h < P),
                    "activation_frequency": float(arr[:, li, h].mean()),
                }
            )

    return layer_rows, head_rows



def learned_router_calibration(
    model,
    batches,
    dev,
    pass_easiness=True,
):
    """Collect per-example, per-layer learned-router counts/targets/errors."""
    rows = []

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            with dev.autocast():
                _ = call_model(
                    model, batch, pass_easiness=pass_easiness
                )
            dev.mark_step()

            ids = cpu_batch["example_id"].tolist()
            easiness = cpu_batch["easiness_score"].tolist()

            for li, block in enumerate(model.model.blocks):
                router = block.mlt_vw_rtr
                actual = (
                    router.save_total_head_count.detach()
                    .to(torch.float32).cpu().tolist()
                )
                target = (
                    router.save_target_total_head_count.detach()
                    .to(torch.float32).cpu().tolist()
                )
                logits = (
                    router.save_router_logits.detach()
                    .to(torch.float32).cpu()
                )
                sig = torch.sigmoid(logits)

                for bi, eid in enumerate(ids):
                    rows.append(
                        {
                            "example_id": int(eid),
                            "layer": li,
                            "easiness_score": float(easiness[bi]),
                            "actual_total_heads": float(actual[bi]),
                            "target_total_heads": float(target[bi]),
                            "count_error": float(actual[bi] - target[bi]),
                            "mean_sigmoid": float(sig[bi].mean()),
                            "sigmoid_saturation_fraction": float(
                                ((sig[bi] < 0.05) | (sig[bi] > 0.95))
                                .float().mean()
                            ),
                            "mean_ste_derivative": float(
                                (sig[bi] * (1.0 - sig[bi])).mean()
                            ),
                        }
                    )

    return rows

# ---------------------------------------------------------------------------
# A_h / I_h common-coalition specialization
# ---------------------------------------------------------------------------

def common_coalition_specialization(
    model,
    examples,
    batches,
    dev,
    masks_by_id,
    variant,
    pass_easiness,
    layers,
    group_size,
    ablation_batch_size,
    seed,
):
    """
    For each elastic/non-permanent head h:
      A_h = examples where baseline policy selected h
      I_h = examples where baseline policy did not select h

    Evaluate BOTH groups under the SAME all-head coalition and remove h.

      utility = CE(all heads except h) - CE(all heads)

    Positive A-I gap means the policy tends to select h on examples where h has
    greater exact single-head utility under the common all-head context.

    For Random-32 this is a negative control: because A/I assignment is random
    wrt input, the expected gap is zero.
    """
    P = int(getattr(model.config, "num_permanent_heads", 0))
    H = int(model.config.num_attention_heads)
    rng = np.random.default_rng(seed)

    Override = (
        RandomAttentionMaskOverride
        if variant == "random"
        else LearnedRouterOverride
    )

    with Override(model, "dense"):
        dense_mean, dense_ce = evaluate_ce(
            model,
            batches,
            dev,
            pass_easiness=pass_easiness,
            return_per_example=True,
        )

    all_ids = sorted(masks_by_id)
    rows = []

    for li in layers:
        for h in range(P, H):
            A = [i for i in all_ids if masks_by_id[i][li, h] > 0.5]
            I = [i for i in all_ids if masks_by_id[i][li, h] <= 0.5]

            rng.shuffle(A)
            rng.shuffle(I)

            nA = min(group_size, len(A))
            nI = min(group_size, len(I))
            nA = (nA // ablation_batch_size) * ablation_batch_size
            nI = (nI // ablation_batch_size) * ablation_batch_size
            A = A[:nA]
            I = I[:nI]

            def eval_group(ids):
                if not ids:
                    return []
                bs = make_subbatches(examples, ids, ablation_batch_size)
                with Override(
                    model,
                    "dense_minus",
                    target_layer=li,
                    target_head=h,
                ):
                    _, vals = evaluate_ce(
                        model,
                        bs,
                        dev,
                        pass_easiness=pass_easiness,
                        return_per_example=True,
                    )
                return [vals[i] - dense_ce[i] for i in ids if i in vals]

            du_A = eval_group(A)
            du_I = eval_group(I)

            rows.append(
                {
                    "layer": li,
                    "head": h,
                    "activation_frequency": float(
                        np.mean([masks_by_id[i][li, h] for i in all_ids])
                    ),
                    "active_examples_used": len(du_A),
                    "inactive_examples_used": len(du_I),
                    "utility_active_mean": safe_mean(du_A),
                    "utility_active_sem": safe_sem(du_A),
                    "utility_inactive_mean": safe_mean(du_I),
                    "utility_inactive_sem": safe_sem(du_I),
                    "specialization_gap_A_minus_I": (
                        safe_mean(du_A) - safe_mean(du_I)
                        if du_A and du_I
                        else float("nan")
                    ),
                }
            )

            print(
                f"L{li:02d} h{h:02d} f="
                f"{rows[-1]['activation_frequency']:.3f} "
                f"A-I={rows[-1]['specialization_gap_A_minus_I']:+.5f}"
            )

    return dense_mean, rows


# ---------------------------------------------------------------------------
# CE mode helpers
# ---------------------------------------------------------------------------

def learned_ce_modes(
    model,
    batches,
    dev,
    pass_easiness,
    random_trials=3,
):
    rows = []

    routed = evaluate_ce(
        model, batches, dev, pass_easiness=pass_easiness
    )
    rows.append({"mode": "learned_routed", "ce": routed, "std": 0.0})

    with LearnedRouterOverride(model, "dense"):
        dense = evaluate_ce(
            model, batches, dev, pass_easiness=pass_easiness
        )
    rows.append({"mode": "forced_dense_all32", "ce": dense, "std": 0.0})

    with LearnedRouterOverride(model, "permanent"):
        perm = evaluate_ce(
            model, batches, dev, pass_easiness=pass_easiness
        )
    rows.append({"mode": "permanent_only", "ce": perm, "std": 0.0})

    vals = []
    for _ in range(int(random_trials)):
        with LearnedRouterOverride(model, "random_same_count"):
            vals.append(
                evaluate_ce(
                    model,
                    batches,
                    dev,
                    pass_easiness=pass_easiness,
                )
            )
    rows.append(
        {
            "mode": "random_same_count",
            "ce": safe_mean(vals),
            "std": safe_std(vals),
        }
    )

    return rows


def random32_ce_modes(
    model,
    batches,
    dev,
    random_trials=5,
):
    rows = []

    # Different random mask draw each trial.
    vals = []
    model.set_routing_mode("random")
    for _ in range(int(random_trials)):
        vals.append(
            evaluate_ce(model, batches, dev, pass_easiness=False)
        )
    rows.append(
        {
            "mode": "random_sparse",
            "ce": safe_mean(vals),
            "std": safe_std(vals),
        }
    )

    with RandomAttentionMaskOverride(model, "dense"):
        dense = evaluate_ce(model, batches, dev, pass_easiness=False)
    rows.append({"mode": "forced_dense_same_bank", "ce": dense, "std": 0.0})

    with RandomAttentionMaskOverride(model, "permanent"):
        perm = evaluate_ce(model, batches, dev, pass_easiness=False)
    rows.append({"mode": "permanent_only", "ce": perm, "std": 0.0})

    return rows


def zip_results(output_dir: Path, name: str):
    # ZIP MUST be outside output_dir to prevent recursive self-zipping.
    zip_base = output_dir.parent / name
    old = zip_base.with_suffix(".zip")
    if old.exists():
        old.unlink()
    return shutil.make_archive(
        str(zip_base), "zip", root_dir=output_dir
    )

Writing helm_analysis_core.py


In [14]:
!python analyze_7e_vs_dense32_attention_anatomy.py \
    --device xla \
    --model-7e model_7e.py \
    --model-dense model_dense32.py \
    --num-examples 16 \
    --batch-size 2 \
    --qk-query-tokens 64 \
    --gradient-probe

E0000 00:00:1786630608.673101    2391 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238
checkpoint-006500.pt: 100%|███████████████| 3.72G/3.72G [00:43<00:00, 86.4MB/s]
training_state.json: 100%|████████████████| 26.1k/26.1k [00:00<00:00, 63.3MB/s]
checkpoint-006500.pt: 100%|███████████████| 3.72G/3.72G [06:24<00:00, 9.67MB/s]
validation-00000.parquet: 100%|███████████| 42.6M/42.6M [00:00<00:00, 56.8MB/s]

=== Activation frequencies ===

=== HELM_7e anatomy ===

=== Dense-32 anatomy ===

=== Current gradient probe ===

Done: /kaggle/working/helm_7e_vs_dense32_attention_anatomy_results.zip
